# Feedback Agent - EDU-MIND

A teaching agent that answers student questions using RAG context.

**LLM:** Groq (Llama 3.1 70B) - Free & Fast

## What This Agent Does
- Receives a student question + relevant documents (from RAG)
- Generates a pedagogical response to teach/explain
- Can also provide feedback after exercise correction

## 1. Setup

In [5]:
# Install deps if needed (run once)
# !uv sync

In [1]:
import os
from dotenv import load_dotenv

# Load .env file
load_dotenv()

# Check API key
if os.getenv("GROQ_API_KEY"):
    print("GROQ_API_KEY loaded")
else:
    print("WARNING: GROQ_API_KEY not found!")
    print("Create a .env file with: GROQ_API_KEY=gsk_your_key_here")

GROQ_API_KEY loaded


In [2]:
from langchain_groq import ChatGroq

def get_llm(temperature: float = 0.7) -> ChatGroq:
    """Create Groq LLM instance."""
    return ChatGroq(
        model="llama-3.3-70b-versatile",  # Current model, free & fast
        api_key=os.getenv("GROQ_API_KEY"),
        temperature=temperature,
    )

# Test connection
try:
    llm = get_llm()
    test = llm.invoke("Dis 'Groq est pret!' en une ligne.")
    print(f"Test: {test.content}")
except Exception as e:
    print(f"Connection failed: {e}")

Test: Le grog est prêt !


## 2. Feedback Agent Prompts

Two variants for testing (Phase 5 - Prompt Evaluation)

In [3]:
# PROMPT V1: Structured pedagogical tutor
PROMPT_V1 = """Tu es un tuteur pedagogique expert et bienveillant.

DOCUMENTS DE REFERENCE:
{context}

QUESTION DE L'ELEVE:
{question}

{correction_info}

INSTRUCTIONS:
- Reponds en te basant UNIQUEMENT sur les documents fournis
- Explique de maniere claire et pedagogique
- Utilise des exemples si necessaire
- Encourage l'eleve
- Si tu ne trouves pas l'information dans les documents, dis-le honnetement

REPONSE:"""

# PROMPT V2: More conversational
PROMPT_V2 = """Tu es un assistant d'etude sympathique qui aide les etudiants a comprendre leurs cours.

Voici les documents que l'etudiant a fournis:
---
{context}
---

L'etudiant demande: {question}

{correction_info}

Reponds de facon naturelle et encourageante, comme un ami qui explique bien. 
Base-toi sur les documents fournis."""

print("Prompts loaded: PROMPT_V1 (structured), PROMPT_V2 (conversational)")

Prompts loaded: PROMPT_V1 (structured), PROMPT_V2 (conversational)


## 3. Main Function

In [4]:
from langchain_core.prompts import ChatPromptTemplate

def generate_feedback(
    question: str,
    context: list[str],
    correction: dict | None = None,
    prompt_version: str = "v1"
) -> str:
    """
    Generate teaching feedback for a student question.
    
    Args:
        question: The student's question
        context: List of relevant documents from RAG
        correction: Optional dict with correction results
                   {"score": 0.5, "errors": ["..."], "is_correct": False}
        prompt_version: "v1" (structured) or "v2" (conversational)
    
    Returns:
        The teaching response as a string
    """
    # Select prompt
    template = PROMPT_V1 if prompt_version == "v1" else PROMPT_V2
    
    # Format context
    context_text = "\n\n".join(context) if context else "Aucun document fourni."
    
    # Format correction info if present
    correction_info = ""
    if correction:
        correction_info = f"""RESULTAT DE L'EXERCICE PRECEDENT:
- Correct: {correction.get('is_correct', 'N/A')}
- Score: {correction.get('score', 'N/A')}
- Erreurs: {', '.join(correction.get('errors', [])) or 'Aucune'}

Prends en compte ce resultat dans ta reponse pour aider l'eleve a comprendre ses erreurs."""
    
    # Create chain
    prompt = ChatPromptTemplate.from_template(template)
    llm = get_llm()
    chain = prompt | llm
    
    # Generate response
    response = chain.invoke({
        "question": question,
        "context": context_text,
        "correction_info": correction_info
    })
    
    return response.content

print("generate_feedback() function ready!")

generate_feedback() function ready!


## 4. Test with Mock Data

In [5]:
# Mock RAG context (simulating retrieved documents)
mock_context = [
    """CHAPITRE 3: LES DERIVEES
La derivee d'une fonction f en un point x mesure le taux de variation instantane.
Formule: f'(x) = lim(h->0) [f(x+h) - f(x)] / h
Regles de base:
- Derivee de x^n = n * x^(n-1)
- Derivee de sin(x) = cos(x)
- Derivee de e^x = e^x""",
    
    """EXEMPLES DE DERIVEES:
1. f(x) = x^2  =>  f'(x) = 2x
2. f(x) = x^3  =>  f'(x) = 3x^2
3. f(x) = 5x^2 + 3x - 2  =>  f'(x) = 10x + 3"""
]

print("Mock context ready (derivatives chapter)")

Mock context ready (derivatives chapter)


In [6]:
# TEST 1: Simple question (no correction)
question1 = "Comment je calcule la derivee de f(x) = x^4 ?"

print("Question:", question1)
print("\n" + "="*50 + "\n")

response1 = generate_feedback(
    question=question1,
    context=mock_context,
    prompt_version="v1"
)

print("Response (V1):")
print(response1)

Question: Comment je calcule la derivee de f(x) = x^4 ?


Response (V1):
**Bienvenue dans notre leçon de calcul des dérivées !**

Je suis ravi de t'aider à comprendre comment calculer la dérivée de la fonction f(x) = x^4.

**Regardons les règles de base**

Selon les documents de référence, nous avons une règle de base pour calculer la dérivée d'une fonction de la forme x^n :

- Derivee de x^n = n * x^(n-1)

**Appliquons cette règle à notre exemple**

Nous voulons calculer la dérivée de f(x) = x^4. En appliquant la règle de base, nous obtenons :

f'(x) = 4 * x^(4-1)
f'(x) = 4 * x^3

**Voilà la réponse !**

La dérivée de f(x) = x^4 est f'(x) = 4x^3.

**Exemples similaires**

Pour te donner une idée plus claire, regardons les exemples de dérivées que nous avons déjà vus :

- f(x) = x^2  =>  f'(x) = 2x
- f(x) = x^3  =>  f'(x) = 3x^2

Tu vois comment la règle de base s'applique dans chaque cas ?

**Encouragement**

Tu as fait un excellent travail en posant cette question ! Le calcul des dér

In [7]:
# TEST 2: After a wrong exercise (with correction)
question2 = "Pourquoi ma reponse etait fausse ?"

mock_correction = {
    "is_correct": False,
    "score": 0.3,
    "errors": [
        "L'eleve a oublie de multiplier par l'exposant",
        "Resultat: x^3 au lieu de 4x^3"
    ]
}

print("Question:", question2)
print("Correction:", mock_correction)
print("\n" + "="*50 + "\n")

response2 = generate_feedback(
    question=question2,
    context=mock_context,
    correction=mock_correction,
    prompt_version="v1"
)

print("Response (V1):")
print(response2)

Question: Pourquoi ma reponse etait fausse ?
Correction: {'is_correct': False, 'score': 0.3, 'errors': ["L'eleve a oublie de multiplier par l'exposant", 'Resultat: x^3 au lieu de 4x^3']}


Response (V1):
**Bienvenue, élève !**

Je suis ravi de vous aider à comprendre vos erreurs et à améliorer vos compétences en calcul de dérivées. Nous allons passer en revue votre exercice précédent et identifier les points à améliorer.

**Analyse de l'exercice**

Vos erreurs ont été identifiées comme suit :

* Vous avez oublié de multiplier par l'exposant.
* Résultat : x^3 au lieu de 4x^3

**Explication de la règle**

La règle de base pour la dérivée d'une fonction de la forme x^n est la suivante :

* La dérivée de x^n = n * x^(n-1)

Cela signifie que lorsque vous dérivez une fonction de la forme x^n, vous devez multiplier l'exposant (n) par la variable (x) élevée à la puissance (n-1).

**Exemple**

Considérons l'exemple suivant :

f(x) = x^3

Pour trouver la dérivée de cette fonction, nous appliquon

In [8]:
# TEST 3: Compare with V2 prompt (conversational)
print("Same question with PROMPT V2 (conversational):\n")

response3 = generate_feedback(
    question=question1,
    context=mock_context,
    prompt_version="v2"
)

print(response3)

Same question with PROMPT V2 (conversational):

Super, on va voir ça ensemble ! 

Pour calculer la dérivée de f(x) = x^4, il faut utiliser la règle de base que l'on a dans les documents : "Derivee de x^n = n * x^(n-1)".

Dans ton cas, n = 4, car ta fonction est x^4. Alors, on applique la règle :

f'(x) = 4 * x^(4-1)
f'(x) = 4 * x^3

Voilà ! La dérivée de f(x) = x^4 est donc f'(x) = 4x^3.

C'est assez simple, n'est-ce pas ? On prend juste le nombre qui est à la puissance (dans ce cas, 4) et on le multiplie par x élevé à la puissance inférieure (dans ce cas, 3).

Si tu veux vérifier, on peut même regarder les exemples donnés dans les documents. On voit que pour f(x) = x^2, la dérivée est 2x, et pour f(x) = x^3, la dérivée est 3x^2. Donc, il est logique que pour f(x) = x^4, la dérivée soit 4x^3 !

J'espère que ça t'a aidé ! Si tu as d'autres questions ou si tu veux vérifier d'autres exemples, n'hésite pas à me demander.
